In [0]:
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
import os
import sys
import logging
from argparse import Namespace

# Automatically detect the root based on where this notebook is sitting
current_path = os.getcwd()
repo_name = "amfs_telemarketing_new_acquisition"

if repo_name in current_path:
    REPO_ROOT = current_path.split(repo_name)[0] + repo_name
else:
    # Fallback if names don't match
    REPO_ROOT = current_path

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
    sys.path.append(os.path.join(REPO_ROOT, "src"))

print(f"✅ Portable REPO_ROOT: {REPO_ROOT}")

# # Add the repo's src directory to the Python path
# REPO_ROOT = os.path.abspath("/Workspace/Users/hanif.m.abidin@gmail.com/amfs_telemarketing_new_acquisition")
# if REPO_ROOT not in sys.path:
#     sys.path.append(REPO_ROOT)

# COMMAND ----------
# Define Widgets for User Input
dbutils.widgets.text("snapshot", "202505", "Snapshot (YYYYMM)")
dbutils.widgets.text("DIL", "DIL1", "DIL Value")
dbutils.widgets.dropdown("mode", "dummy", ["dummy", "real"], "Training Mode")

# Retrieve Parameters
SNAPSHOT = dbutils.widgets.get("snapshot")
DIL = dbutils.widgets.get("DIL")
MODE = dbutils.widgets.get("mode")



from lib.train.train_prep import prepare_training_data
from lib.train.model_trainer import train_model

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define Configuration Paths
CONFIG_DIR = os.path.join(REPO_ROOT, "configs")
MAIN_CONFIG = os.path.join(CONFIG_DIR, "main_config.yaml")
TRAIN_CONFIG = os.path.join(CONFIG_DIR, "train_config.yaml")
MODEL_CONFIG = os.path.join(CONFIG_DIR, "model_config.yaml")

# Create mock args object
args = Namespace(snapshot=SNAPSHOT, dil=DIL, mode=MODE)

# COMMAND ----------
import os
import sys
import logging
from argparse import Namespace

# Add the repo's src directory to the Python path
REPO_ROOT = os.path.abspath("/Workspace/Users/hanif.m.abidin@gmail.com/amfs_telemarketing_new_acquisition")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from lib.train.train_prep import prepare_training_data
from lib.train.model_trainer import train_model

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Define Configuration Paths
CONFIG_DIR = os.path.join(REPO_ROOT, "configs")
MAIN_CONFIG = os.path.join(CONFIG_DIR, "main_config.yaml")
TRAIN_CONFIG = os.path.join(CONFIG_DIR, "train_config.yaml")
MODEL_CONFIG = os.path.join(CONFIG_DIR, "model_config.yaml")

# Create mock args object
args = Namespace(snapshot=SNAPSHOT, dil=DIL, mode=MODE)

# COMMAND ----------
# DBTITLE 1,Step 1: Data Preparation & Target Labeling
logging.info(f"Starting Training Data Preparation in {MODE} mode...")

# This handles multi-snapshot stacking if MODE is 'real'
train_data_path = prepare_training_data(MAIN_CONFIG, TRAIN_CONFIG, args)

print(f"✅ Training data prepared and saved to: {train_data_path}")

# COMMAND ----------
# DBTITLE 1,Step 2: Model Training (XGBoost)
logging.info(f"Starting Model Training for snapshot {SNAPSHOT}...")

# Fits the model, logs ROC-AUC, and saves the pickle file
model_pkl_path = train_model(MAIN_CONFIG, TRAIN_CONFIG, MODEL_CONFIG, args)

print(f"🚀 Model training complete!")
print(f"Model File: {model_pkl_path}")

In [0]:
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
# COMMAND ----------
# DBTITLE 1,Step 0: Infrastructure Setup (Auto-Bootstrap)
# Define Names
CATALOG_NAME = "main"
SCHEMA_NAME = "default"
VOLUME_NAME = "amfs_telemarketing_new_acquisition"

# Create infrastructure via SQL
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}")

# COMMAND ----------
# DBTITLE 1,Step 1: User Parameters
dbutils.widgets.text("snapshot", "202505", "Snapshot (YYYYMM)")
dbutils.widgets.text("DIL", "70", "DIL Value")
dbutils.widgets.dropdown("mode", "dummy", ["dummy", "real"], "Training Mode")

SNAPSHOT = dbutils.widgets.get("snapshot")
DIL = dbutils.widgets.get("DIL")
MODE = dbutils.widgets.get("mode")


# COMMAND ----------
# DBTITLE 1,Step 2: Portable Path Logic
import os
import sys

# Automatically detect the root based on where this notebook is sitting
current_path = os.getcwd()
repo_name = "amfs_telemarketing_new_acquisition"

if repo_name in current_path:
    REPO_ROOT = current_path.split(repo_name)[0] + repo_name
else:
    # Fallback if names don't match
    REPO_ROOT = current_path

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)
    sys.path.append(os.path.join(REPO_ROOT, "src"))

print(f"✅ Portable REPO_ROOT: {REPO_ROOT}")

# Imports after path is set
import logging
from argparse import Namespace
from lib.train.train_prep import prepare_training_data
from lib.train.model_trainer import train_model

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# COMMAND ----------
# DBTITLE 1,Step 3: Run Training Pipeline
# Define Config Paths
CONFIG_DIR = os.path.join(REPO_ROOT, "configs")
MAIN_CONFIG = os.path.join(CONFIG_DIR, "main_config.yaml")
TRAIN_CONFIG = os.path.join(CONFIG_DIR, "train_config.yaml")
MODEL_CONFIG = os.path.join(CONFIG_DIR, "model_config.yaml")

# Create mock args
args = Namespace(snapshot=SNAPSHOT, dil=DIL, mode=MODE)

# --- 3a. Prepare Training Data ---
logging.info(f"Preparing training data for {SNAPSHOT}...")
train_data_path = prepare_training_data(MAIN_CONFIG, TRAIN_CONFIG, args)

# --- 3b. Train Model ---
logging.info(f"Training model in {MODE} mode...")
model_pkl_path = train_model(MAIN_CONFIG, TRAIN_CONFIG, MODEL_CONFIG, args)

print(f"✨ Training finished!")
print(f"Matrix: {train_data_path}")
print(f"Model: {model_pkl_path}")